# Descarga de los datos agrícolas

Esta libreta documenta la obtención y preparación de los datos agrícolas
utilizados para estudiar los factores asociados con el éxito de la
producción agrícola en los municipios de México entre **2003 y 2025**.

Los archivos agrícolas se obtienen anualmente de la base de datos abiertos de
la Secretaría de Agricultura y posteriormente se integran en un único
conjunto histórico.

Debido a que la estructura de los archivos presenta algunos cambios a
lo largo del periodo de estudio, el procesamiento incluye una etapa de
normalización para mantener un esquema consistente entre los años.

La lógica de descarga y transformación se encuentra implementada en el
paquete `proyecto_agronomia`. Esta libreta permite ejecutar, inspeccionar
y validar el proceso de manera reproducible.

### Periodo de estudio

**2003–2025**

### Flujo de trabajo

1. Descargar los archivos agrícolas anuales.
2. Verificar la disponibilidad de los archivos.
3. Inspeccionar un archivo agrícola.
4. Normalizar las diferencias históricas del esquema.
5. Construir el conjunto agrícola histórico.
6. Generar variables derivadas e indicadores de calidad.
7. Validar el conjunto resultante.

In [1]:
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIO_INICIAL,
    ANIO_FINAL,
    ANIOS,
)

from proyecto_agronomia.dataset import (
    descargar_datos_agricolas,
    descargar_diccionario_agricola,
)

2026-11-20 23:14:02.015 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## Descarga de los datos agrícolas

La fuente agrícola proporciona un archivo independiente para cada año
del periodo de estudio.

Para construir la serie histórica se consideran los años comprendidos
entre **2003 y 2025**.

Los archivos originales se almacenan en:

`data/raw/agricultura/`

Los datos de esta carpeta representan la información original obtenida
de la fuente y no son modificados directamente durante las etapas
posteriores del procesamiento.

La función de descarga verifica si cada archivo ya existe localmente.
Cuando un archivo está disponible, se reutiliza en lugar de volver a
descargarlo.

In [2]:
print(
    f"Periodo de estudio: "
    f"{ANIO_INICIAL}-{ANIO_FINAL}"
)

print(
    f"Años considerados: "
    f"{len(ANIOS)}"
)

print(
    f"Directorio de datos originales: "
    f"{AGRICULTURA_RAW_DIR}"
)

Periodo de estudio: 2003-2025
Años considerados: 23
Directorio de datos originales: B:\MCD2\JW\agro\proyecto_agronomia\data\raw\agricultura


## Ejecución de la descarga

Se utiliza la función `descargar_datos_agricolas()` del módulo
`proyecto_agronomia.dataset`.

La función recorre los años del periodo de estudio y conserva los
archivos que ya se encuentran disponibles localmente, por lo que la
operación puede ejecutarse nuevamente sin descargar innecesariamente
los mismos datos.

In [3]:
archivos_agricolas = descargar_datos_agricolas(
    ANIOS
)

print(
    f"Archivos disponibles: "
    f"{len(archivos_agricolas)}"
)

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

... (16 outputs left out)

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

Datos agrícolas: 100%|██████████| 23/23 [00:00<00:00, 248.94it/s]

2026-11-20 23:14:02.799 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2003] El archivo ya existe.
2026-11-20 23:14:02.802 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2004] El archivo ya existe.
2026-11-20 23:14:02.807 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2005] El archivo ya existe.
2026-11-20 23:14:02.810 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2006] El archivo ya existe.
2026-11-20 23:14:02.815 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2007] El archivo ya existe.
2026-11-20 23:14:02.819 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2008] El archivo ya existe.
2026-11-20 23:14:02.823 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:70 - [2009] El archivo ya existe.
2026-11-20 23:14:02.827 | INFO     | proy

### Verificación de los archivos

Antes de iniciar el procesamiento se verifica que exista un archivo
para cada uno de los años considerados en el estudio.

Esta validación permite detectar años faltantes antes de construir el
conjunto histórico. Por defecto esta de 2003 a 2025, entonces esperamos 23 archivos

In [4]:
disponibilidad = pd.DataFrame({
        "Anio": ANIOS,
        "Archivo correspondiente": [
            f"agricola_{anio}.csv"
            for anio in ANIOS if (AGRICULTURA_RAW_DIR / f"agricola_{anio}.csv").exists()
        ]
    }
)

disponibilidad

,Anio,Archivo correspondiente
0,2003,agricola_2003.csv
1,2004,agricola_2004.csv
2,2005,agricola_2005.csv
3,2006,agricola_2006.csv
4,2007,agricola_2007.csv
5,2008,agricola_2008.csv
6,2009,agricola_2009.csv
7,2010,agricola_2010.csv
8,2011,agricola_2011.csv
9,2012,agricola_2012.csv


In [5]:
print(
    f"Archivos esperados: "
    f"{len(ANIOS)}"
)

print(
    f"Archivos disponibles: "
    f"{disponibilidad.shape[0]}"
)

print(
    f"Archivos faltantes: "
    f"{ ANIO_FINAL - ANIO_INICIAL + 1 - disponibilidad.shape[0]}"
)

Archivos esperados: 23
Archivos disponibles: 23
Archivos faltantes: 0


### Diccionario de Datos

Siempre al descargar los datos. Debemos adjuntar su Diccionario de Datos.

In [6]:
ruta_diccionario = descargar_diccionario_agricola()

print(
    f"Diccionario disponible en: "
    f"{ruta_diccionario}"
)

2026-11-20 23:14:02.943 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_diccionario_agricola:130 - El diccionario agrícola ya existe.
Diccionario disponible en: B:\MCD2\JW\agro\proyecto_agronomia\references\diccionario_datos_agricultura_dgsiap.xlsx
